# 01 - Pré-processamento

Pré-processamento do ficheiro *accepted* (Fase 2 do enunciado, *Pre-processing*).

**Ordem dos passos**

O enunciado pede: *"Define partitions before data-dependent preprocessing"*. Por isso, antes de olhar para valores em falta, outliers ou qualidade dos dados, faz-se só o que não depende de nenhuma estatística dos dados:
1. Remover as linhas que não são empréstimos (regra estrutural).
2. Separar as partições temporais definidas na Fase 1 (`docs/01_formulacao_problema/desenho_split.md`).

A partir daqui, todas as análises e decisões de limpeza usam **só o conjunto de desenvolvimento**. O teste final fica guardado à parte e só recebe as mesmas regras, sem as rever.

**Antes de correr:** é preciso ter corrido `00_conversao_dados.ipynb`. O ficheiro *accepted* ocupa cerca de 4 GB em memória.

In [ ]:
# Setup: imports e caminhos do projeto
import gc
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import pandas as pd

from lending import config
from lending.cleaning import find_non_loan_rows
from lending.exclusions import ExclusionLog
from lending.parsing import parse_month_year
from lending.partition import TemporalPartitioner
from lending.schema import SchemaParser, SchemaReport
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Linhas que não são empréstimos

O CSV do Kaggle é a junção de várias exportações da LendingClub. Cada exportação termina com duas linhas de totais (`Total amount funded in policy code 1/2: ...`) e há também um separador (`Loans that do not meet the credit policy`). Estas linhas estão **a meio do ficheiro**, não só no fim.

**Regra:** o `id` não é um número inteiro **e** todas as outras colunas estão vazias. Se aparecer um `id` não numérico numa linha com dados, o código dá erro em vez de a remover, porque já não seria um rodapé.

A regra é estrutural, por isso aplica-se ao ficheiro inteiro, antes da partição. Cada remoção fica no registo de exclusões, que no fim dá a tabela de exclusões do relatório.

In [ ]:
accepted = PickleStore.load(config.ACCEPTED_RAW_PKL, expected_type=pd.DataFrame)
print(f"accepted: {len(accepted):,} linhas x {accepted.shape[1]} colunas")

non_loan = find_non_loan_rows(accepted)
print(f"Linhas que não são empréstimos: {non_loan.sum()}")

# O texto de cada linha marcada e quantas vezes aparece
accepted.loc[non_loan, "id"].str.replace(r"\d+$", "<valor>", regex=True).value_counts()

In [ ]:
# Posição das linhas no ficheiro: confirma que estão espalhadas, entre exportações
accepted.loc[non_loan, ["id"]]

In [ ]:
log = ExclusionLog("accepted -> desenvolvimento")

# Só se regista: filtrar o DataFrame inteiro criaria uma cópia de ~4 GB.
# As linhas que ficam são escolhidas na partição (passo 2).
log.record(
    step="Linhas que não são empréstimos",
    kind="remoção",
    rule="id não numérico e todas as outras colunas vazias",
    reason="Totais e separadores das exportações da LendingClub, não são empréstimos",
    rows_before=len(accepted),
    rows_removed=int(non_loan.sum()),
)
log.to_frame()

## 2. Partição temporal

**Data usada:** `issue_d`, o mês de financiamento (texto `Mon-AAAA`, ex.: `Dec-2015`). A conversão para data é **determinística**: uma tabela fixa de meses, sem estimar nada a partir dos dados. Por isso pode ser feita antes da partição (é aliás necessária para a fazer). Um valor fora do formato dá erro.

**Cortes** (`config.PARTITIONS`, decididos na Fase 1):

| Partição | Período (`issue_d`) | Utilização |
|---|---|---|
| Desenvolvimento | 2007-06 a 2014-12 | Treino, validação, EDA, testes de hipóteses, *fit* das transformações |
| Teste final | 2015-01 a 2015-12 | Avaliação final, uma única vez, no fim da Parte 2 |
| Reservado | 2016-01 a 2018-12 | Não usado na Parte 1 |

Os cortes são feitos ao mês, por isso os empréstimos do mesmo mês ficam sempre juntos. Cada linha tem de cair numa e uma só partição: uma data em falta ou fora dos períodos dá erro.

In [ ]:
# Datas só das linhas que são empréstimos
issue_date = parse_month_year(accepted.loc[~non_loan, "issue_d"])
print(f"issue_d: {issue_date.min():%Y-%m} a {issue_date.max():%Y-%m}, "
      f"{issue_date.isna().sum()} em falta")

partitioner = TemporalPartitioner()
partitioner.summary(issue_date)

In [ ]:
# Empréstimos por ano de financiamento e partição
# (só contagens: não se olha para nenhum valor do teste nem do reservado)
pd.crosstab(issue_date.dt.year.rename("ano"), partitioner.assign(issue_date).rename("partição"),
            margins=True, margins_name="Total")

In [ ]:
# Só se copiam para memória as partições usadas; o reservado continua no pickle original
parts = partitioner.split(accepted, issue_date, names=["development", "test"])
accepted_dev, accepted_test = parts["development"], parts["test"]

sizes = partitioner.summary(issue_date)["rows"]
assert len(accepted_dev) == sizes["development"] and len(accepted_test) == sizes["test"]

log.record(
    step="Separar o teste final",
    kind="partição",
    rule="issue_d de 2015-01 a 2015-12",
    reason="Teste final cronológico, guardado à parte para a avaliação final (desenho_split.md)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["test"]),
)
log.record(
    step="Separar o período reservado",
    kind="partição",
    rule="issue_d de 2016-01 a 2018-12",
    reason="Não usado na Parte 1: os empréstimos de 36 meses ainda não tinham terminado em 2018-12",
    rows_before=log.rows_after,
    rows_removed=int(sizes["reserved"]),
)
assert log.rows_after == len(accepted_dev)

# Libertar o ficheiro completo: a partir daqui só se usa o desenvolvimento
del accepted, parts, non_loan, issue_date
_ = gc.collect()

log.to_frame()

## 3. Guardar as partições

As partições são guardadas **ainda em texto**, sem nenhum valor alterado: este passo só separa linhas. A conversão de tipos e a limpeza vêm a seguir, sobre o desenvolvimento.

- `accepted_dev_raw.pkl`: conjunto de desenvolvimento.
- `accepted_test_raw.pkl`: teste final. **Não volta a ser aberto** até à avaliação final; nessa altura recebe as mesmas regras de limpeza, sem as rever.
- `accepted_dev_exclusions.pkl`: o registo de exclusões, que os passos seguintes continuam.

O período reservado não é guardado: não é usado na Parte 1 e continua disponível em `accepted_raw.pkl`.

In [ ]:
PickleStore.save(accepted_dev, config.ACCEPTED_DEV_RAW_PKL)
PickleStore.save(accepted_test, config.ACCEPTED_TEST_RAW_PKL)
PickleStore.save(log, config.ACCEPTED_DEV_LOG_PKL)

print(f"Desenvolvimento: {len(accepted_dev):,} linhas -> {config.ACCEPTED_DEV_RAW_PKL.name}")
print(f"Teste final:     {len(accepted_test):,} linhas -> {config.ACCEPTED_TEST_RAW_PKL.name}")
print(f"Registo:         {log} -> {config.ACCEPTED_DEV_LOG_PKL.name}")

# O teste sai da memória: a partir daqui não é usado
del accepted_test
_ = gc.collect()

## 4. Conversão de tipos (parsing determinístico)

Até aqui todas as colunas são texto. Cada coluna é agora convertida para o seu tipo, segundo o schema declarado em `src/lending/schema.py`, onde o tipo de cada uma das 151 colunas está escrito à mão em vez de ser adivinhado pelo pandas.

| Tipo | Exemplo no ficheiro | Resultado | Colunas |
|---|---|---|---:|
| `id` | `"9006963"` | inteiro | 1 |
| `number` | `"10000.0"`, `"12.99"`, `"1e-10"` | decimal (`float64`) | 112 |
| `term` | `" 36 months"` | `36.0` (meses) | 1 |
| `emp_length` | `"< 1 year"`, `"10+ years"` | `0` a `10` (anos) | 1 |
| `month_year` | `"Dec-2015"` | `2015-12-01` | 11 |
| `flag` | `"Y"`/`"N"`, `"y"`/`"n"` | `True`/`False` | 3 |
| `category` | `"MORTGAGE"`, `"B3"` | texto sem espaços nas pontas | 17 |
| `text` | `"Teacher"`, url, `desc` | texto sem espaços nas pontas | 5 |

**Porque pode ser feito antes de qualquer *fit*:** cada valor é convertido só a partir de si próprio, com uma regra fixa. Nada é estimado a partir dos dados, por isso o mesmo `SchemaParser` vai converter o teste final exatamente da mesma forma, sem o rever. É o caso que o enunciado dá como exemplo: *"removing a percent sign or converting a date string using a fixed format"*.

**Garantias:**
- Um valor com formato inesperado dá **erro**, nunca passa a NaN em silêncio.
- Nenhum valor preenchido se perde. A única exceção é o texto só com espaços, que é um campo vazio e passa a valor em falta (fica contado na tabela abaixo).
- Não se mudam maiúsculas nem se juntam categorias (ex.: `"RN"` e `"Registered Nurse"`): isso é uma decisão de limpeza, não de parsing.

In [ ]:
parser = SchemaParser()
accepted_dev_parsed = parser.parse(accepted_dev)
parse_summary = parser.summary(accepted_dev, accepted_dev_parsed)

print(f"{accepted_dev_parsed.shape[1]} colunas convertidas | memória: "
      f"{accepted_dev.memory_usage(deep=True).sum() / 1e6:,.0f} MB -> "
      f"{accepted_dev_parsed.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

# Quantas colunas de cada tipo e com que dtype ficaram
parse_summary.groupby(["kind", "dtype"]).size().rename("colunas").to_frame()

In [ ]:
# Colunas em que algum valor passou a valor em falta (só pode ser texto só com espaços)
parse_summary[parse_summary["blank_to_missing"] > 0]

In [ ]:
# Antes e depois, nas mesmas linhas (amostra aleatória com semente, só para ver a conversão)
example_columns = ["id", "term", "int_rate", "emp_length", "issue_d", "earliest_cr_line",
                   "revol_util", "pymnt_plan", "home_ownership"]
example_rows = accepted_dev.sample(5, random_state=config.RANDOM_SEED).index

pd.concat({"antes (texto)": accepted_dev.loc[example_rows, example_columns],
           "depois": accepted_dev_parsed.loc[example_rows, example_columns]}, axis=1)

In [ ]:
# As tabelas de conversão de term e emp_length: cada texto corresponde a um só número
# (dropna=False mostra também os valores em falta, que continuam em falta)
for col in ["term", "emp_length"]:
    mapping = (pd.DataFrame({"texto": accepted_dev[col], "valor": accepted_dev_parsed[col]})
               .value_counts(dropna=False).rename("linhas").reset_index()
               .sort_values("valor", na_position="last"))
    display(mapping.set_index("texto"))

In [ ]:
# A partir daqui trabalha-se com o desenvolvimento já convertido
# (a versão em texto continua guardada em accepted_dev_raw.pkl)
accepted_dev = accepted_dev_parsed
del accepted_dev_parsed
_ = gc.collect()

accepted_dev.dtypes.value_counts().rename("colunas").to_frame()

## 5. Schema e unidades

O enunciado pede para descrever *"the source files, schema, units"*. O **significado** de cada coluna já está na tabela de elegibilidade da Fase 1 (`docs/01_formulacao_problema/elegibilidade_colunas.md`). Aqui descreve-se, para cada coluna:
- o **tipo** depois da conversão e o `dtype`;
- a **unidade** (`ACCEPTED_UNITS` em `src/lending/schema.py`): USD, %, meses, pontos FICO, contagem, etc.;
- o **intervalo observado** no desenvolvimento, que serve para confirmar a unidade. Exemplo: `int_rate` entre 5,42 e 26,06 confirma que está em percentagem (12,99 = 12,99%) e não em fração de 0 a 1.

**Convenção das percentagens:** todas as colunas em % estão na escala 0–100, não 0–1. Algumas passam de 100 (ex.: `revol_util`), o que é analisado na qualidade dos dados.

A tabela completa é escrita em `docs/02_preprocessamento/schema_accepted.md`.

In [ ]:
schema_report = SchemaReport()
schema_description = schema_report.describe(accepted_dev)

# Quantas colunas há de cada unidade
schema_description["unit"].value_counts().rename("colunas").to_frame()

In [ ]:
# Confirmação das unidades: colunas em % e pontos FICO, com o intervalo observado
schema_description[schema_description["unit"].isin(["%", "pontos FICO"])
                   & (schema_description["distinct"] > 0)]

In [ ]:
# Datas: a última data registada em cada coluna mostra até quando o ficheiro foi atualizado
date_columns = schema_description.index[schema_description["kind"] == "month_year"]
last_dates = accepted_dev[date_columns].max().dt.strftime("%Y-%m").rename("última data")
last_dates.to_frame().assign(depois_de_2018=accepted_dev[date_columns].gt("2018-12-31").sum())

In [ ]:
# Tabela completa do schema (151 colunas) e registo em docs/
out = schema_report.write_markdown(schema_description,
                                   config.DOCS_DIR / "02_preprocessamento" / "schema_accepted.md",
                                   dataset="accepted, desenvolvimento", n_rows=len(accepted_dev))
print("Escrito:", out.relative_to(config.PROJECT_ROOT))

with pd.option_context("display.max_rows", 200):
    display(schema_description)